# Case Study: Clustering & Outlier Detection

Explore clustering algorithms and anomaly detection techniques on retail transaction data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

sns.set_style('whitegrid')
%matplotlib inline
print('Ready!')


## 1. Load Customer Transaction Data


In [ ]:
df = pd.read_csv('Data/fraud_data.csv')
print('Shape:', df.shape)
df.head()


In [ ]:
features = ['amount', 'distance_from_home', 'transaction_hour', 'previous_attempts_24h']
X = df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 2. K-Means Clustering


In [ ]:
# Find optimal K using elbow method
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal K')
plt.grid(True)
plt.show()


In [ ]:
# Apply K-Means with optimal K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

# Visualize with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['kmeans_cluster'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.title('K-Means Clusters (PCA-reduced view)')
plt.show()


## 3. Cluster Analysis


In [ ]:
df.groupby('kmeans_cluster')[features].mean().round(2)


## 4. DBSCAN for Density-Based Clustering


In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=10)
df['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

print('DBSCAN clusters found:', df['dbscan_cluster'].nunique() - 1)
print('Points labeled as noise (-1):', (df['dbscan_cluster'] == -1).sum())

plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['dbscan_cluster'], cmap='plasma', alpha=0.6)
plt.colorbar(scatter, label='DBSCAN Cluster')
plt.title('DBSCAN Clustering')
plt.show()


## 5. Outlier Detection – Isolation Forest


In [ ]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)
df['iforest_anomaly'] = iso_forest.fit_predict(X_scaled)
df['iforest_score'] = iso_forest.score_samples(X_scaled)

anomalies_if = df[df['iforest_anomaly'] == -1]
print(f'Isolation Forest detected {len(anomalies_if)} anomalies')

plt.figure(figsize=(8, 6))
colors = df['iforest_anomaly'].map({1: 'blue', -1: 'red'})
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6)
plt.title('Isolation Forest – Anomalies in Red')
plt.show()


## 6. Outlier Detection – Local Outlier Factor


In [ ]:
lof = LocalOutlierFactor(contamination=0.05)
df['lof_anomaly'] = lof.fit_predict(X_scaled)

anomalies_lof = df[df['lof_anomaly'] == -1]
print(f'LOF detected {len(anomalies_lof)} anomalies')

plt.figure(figsize=(8, 6))
colors = df['lof_anomaly'].map({1: 'blue', -1: 'red'})
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.6)
plt.title('LOF – Anomalies in Red')
plt.show()


## 7. Overlap Analysis


In [ ]:
overlap = df[(df['iforest_anomaly'] == -1) & (df['lof_anomaly'] == -1)]
print(f'Both methods agree on {len(overlap)} anomalies')
overlap[features].describe()


## Summary


In [ ]:
print('''
Key Learnings:
- K-Means finds spherical clusters; use elbow method for K selection
- DBSCAN detects arbitrary-shaped clusters and noise points
- Isolation Forest and LOF are powerful for unsupervised anomaly detection
- Combining multiple detection methods gives more robust results
- Always scale features before clustering / anomaly detection
''')
